In [16]:
# CELL 1 - READ BRONZE TABLE

bronze_patients_df  = spark.table("bronze_patients")

print("Bronze Rows:", bronze_patients_df.count())
bronze_patients_df.printSchema()
print(bronze_patients_df.columns)

display(bronze_patients_df.limit(20))

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 18, Finished, Available, Finished, False)

Bronze Rows: 540743
root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- home_address: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- policy_number: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- last_updated_timestamp: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'email', 'phone_number', 'home_address', 'insurance_provider', 'policy_number', 'is_active', 'start_dat

SynapseWidget(Synapse.DataFrame, 0027ba97-3983-4ede-a50d-59c778d0b962)

In [17]:
# CELL 2 — Standardize patient column names

column_mapping = {
    "first_name_": "first_name",
    "date_of_bi": "date_of_birth",
    "gender_": "gender",
    "email_": "email",
    "phone_numb": "phone_number",
    "is_active_": "is_active",
    "last_updat": "last_updated_timestamp"
}

silver_patients_df = bronze_patients_df

for old_name, new_name in column_mapping.items():
    if old_name in silver_patients_df.columns:
        silver_patients_df = silver_patients_df.withColumnRenamed(
            old_name,
            new_name
        )

silver_patients_df.printSchema()
print(silver_patients_df.columns)
display(silver_patients_df.limit(20))

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 19, Finished, Available, Finished, False)

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- home_address: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- policy_number: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- last_updated_timestamp: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'email', 'phone_number', 'home_address', 'insurance_provider', 'policy_number', 'is_active', 'start_date', 'end_date', 'las

SynapseWidget(Synapse.DataFrame, c919ce9b-c5e5-47af-8a2b-eedbc2cdd5dc)

In [18]:
# CELL 3 — BASIC STRING CLEANING
# Fully rerunnable
# Purpose:
#   1. Trim leading and trailing spaces
#   2. Convert placeholder/null-like strings to actual NULL
#   3. Preserve valid capitalization and original values
# No tables are written in this cell

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    when
)

# Generic placeholder values that should become actual NULL
null_like_values = [
    "",
    "null",
    "none",
    "n/a",
    "na",
    "no value",
    "undefined",
    "unknown",
    "???",
    "??",
    "#"
]

# Apply only to string columns
for column_name, data_type in silver_patients_df.dtypes:

    if data_type == "string":

        silver_patients_df = silver_patients_df.withColumn(
            column_name,
            when(
                col(column_name).isNull(),
                None
            )
            .when(
                lower(trim(col(column_name))).isin(
                    null_like_values
                ),
                None
            )
            .otherwise(
                trim(col(column_name))
            )
        )

print("SUCCESS: Basic patient string cleaning completed.")

display(
    silver_patients_df.limit(20)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 20, Finished, Available, Finished, False)

SUCCESS: Basic patient string cleaning completed.


SynapseWidget(Synapse.DataFrame, 3174b98c-ac30-4ea8-ae9d-90df2f453d9c)

In [19]:
# CELL 4 — STANDARDIZE PATIENT BUSINESS VALUES
# Fully rerunnable
# Standardizes:
#   - first_name and last_name capitalization
#   - gender
#   - is_active
#   - insurance_provider
# No tables are written in this cell

from pyspark.sql.functions import (
    col,
    lower,
    trim,
    when,
    initcap
)

silver_patients_df = (
    silver_patients_df

    # ---------------------------------------------------------
    # STANDARDIZE PATIENT NAMES
    # Examples:
    #   JASON -> Jason
    #   alexis -> Alexis
    #   DAVIS -> Davis
    # ---------------------------------------------------------

    .withColumn(
        "first_name",
        when(
            col("first_name").isNotNull(),
            initcap(lower(trim(col("first_name"))))
        ).otherwise(None)
    )

    .withColumn(
        "last_name",
        when(
            col("last_name").isNotNull(),
            initcap(lower(trim(col("last_name"))))
        ).otherwise(None)
    )

    # ---------------------------------------------------------
    # STANDARDIZE GENDER
    # ---------------------------------------------------------

    .withColumn(
        "gender",
        when(
            lower(trim(col("gender"))).isin(
                "m",
                "male"
            ),
            "M"
        )
        .when(
            lower(trim(col("gender"))).isin(
                "f",
                "female"
            ),
            "F"
        )
        .when(
            lower(trim(col("gender"))).isin(
                "other",
                "non-binary",
                "nonbinary"
            ),
            "Other"
        )
        .otherwise(None)
    )

    # ---------------------------------------------------------
    # STANDARDIZE ACTIVE STATUS
    # ---------------------------------------------------------

    .withColumn(
        "is_active",
        when(
            lower(trim(col("is_active"))).isin(
                "true",
                "yes",
                "y",
                "1",
                "active"
            ),
            True
        )
        .when(
            lower(trim(col("is_active"))).isin(
                "false",
                "no",
                "n",
                "0",
                "inactive"
            ),
            False
        )
        .otherwise(None)
    )

    # ---------------------------------------------------------
    # STANDARDIZE INSURANCE PROVIDER
    # Invalid providers remain unchanged so Cell 8 can
    # quarantine them using the approved-provider list.
    # ---------------------------------------------------------

    .withColumn(
        "insurance_provider",
        when(
            lower(trim(col("insurance_provider")))
            == "sun life",
            "Sun Life"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "manulife",
            "Manulife"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "canada life",
            "Canada Life"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "blue cross",
            "Blue Cross"
        )
        .when(
            lower(trim(col("insurance_provider"))).isin(
                "desjardins insurance",
                "desjardins insurances"
            ),
            "Desjardins Insurance"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "ohip (ontario)",
            "OHIP (Ontario)"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "ahcip (alberta)",
            "AHCIP (Alberta)"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "msp (british columbia)",
            "MSP (British Columbia)"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "ramq (quebec)",
            "RAMQ (Quebec)"
        )
        .when(
            lower(trim(col("insurance_provider")))
            == "msi (nova scotia)",
            "MSI (Nova Scotia)"
        )
        .otherwise(trim(col("insurance_provider")))
    )
)

print("SUCCESS: Patient business values standardized.")

display(
    silver_patients_df.select(
        "patient_id",
        "first_name",
        "last_name",
        "gender",
        "insurance_provider",
        "is_active"
    ).limit(50)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 21, Finished, Available, Finished, False)

SUCCESS: Patient business values standardized.


SynapseWidget(Synapse.DataFrame, 23a67d15-bf28-466a-b5af-88c29403f70c)

In [20]:
# CELL 5 — Convert valid dates/timestamps and null invalid values

from pyspark.sql.functions import col, to_date, to_timestamp, when

silver_patients_df = (
    silver_patients_df

    # Date of birth
    .withColumn(
        "date_of_birth",
        when(
            to_date(col("date_of_birth"), "yyyy-MM-dd").isNotNull(),
            to_date(col("date_of_birth"), "yyyy-MM-dd")
        ).otherwise(None)
    )

    # Start date
    .withColumn(
        "start_date",
        when(
            to_date(col("start_date"), "yyyy-MM-dd").isNotNull(),
            to_date(col("start_date"), "yyyy-MM-dd")
        ).otherwise(None)
    )

    # End date
    .withColumn(
        "end_date",
        when(
            to_date(col("end_date"), "yyyy-MM-dd").isNotNull(),
            to_date(col("end_date"), "yyyy-MM-dd")
        ).otherwise(None)
    )

    # Last updated timestamp
    .withColumn(
        "last_updated_timestamp",
        when(
            to_timestamp(col("last_updated_timestamp")).isNotNull(),
            to_timestamp(col("last_updated_timestamp"))
        ).otherwise(None)
    )
)

display(
    silver_patients_df.select(
        "patient_id",
        "date_of_birth",
        "start_date",
        "end_date",
        "last_updated_timestamp"
    ).limit(50)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3f52d8e-308c-4912-8fd8-db091d201b38)

In [21]:
# CELL 6 — VALIDATE CORE PATIENT FIELDS
# Fully rerunnable: standardizes valid values and nulls invalid formats

from pyspark.sql.functions import (
    col,
    lower,
    upper,
    when,
    regexp_replace,
    trim
)

# Digits-only version used for phone validation
phone_digits = regexp_replace(
    col("phone_number"),
    r"\D",
    ""
)

silver_patients_df = (
    silver_patients_df

    # ---------------------------------------------------------
    # PATIENT ID
    # ---------------------------------------------------------

    .withColumn(
        "patient_id",
        when(
            upper(trim(col("patient_id"))).rlike(
                r"^PAT_[0-9]{6}$"
            ),
            upper(trim(col("patient_id")))
        ).otherwise(None)
    )

    # ---------------------------------------------------------
    # EMAIL
    # ---------------------------------------------------------

    .withColumn(
        "email",
        when(
            lower(trim(col("email"))).rlike(
                r"^[A-Za-z0-9._%+-]+@"
                r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
            ),
            lower(trim(col("email")))
        ).otherwise(None)
    )

    # ---------------------------------------------------------
    # PHONE NUMBER
    # Accept common separators, but reject:
    # - extensions such as x912 or ext 42
    # - all-zero values
    # - fewer than 10 or more than 15 digits
    # ---------------------------------------------------------

    .withColumn(
        "phone_number",
        when(
            col("phone_number").isNotNull()

            # Allowed visible characters only
            & trim(col("phone_number")).rlike(
                r"^\+?[0-9().\-\s]+$"
            )

            # No extension suffix
            & ~lower(trim(col("phone_number"))).rlike(
                r"(x|ext\.?|extension)\s*[0-9]+$"
            )

            # Total digit length must be 10–15
            & phone_digits.rlike(
                r"^[0-9]{10,15}$"
            )

            # Reject all-zero phone numbers
            & ~phone_digits.rlike(
                r"^0+$"
            ),

            trim(col("phone_number"))
        ).otherwise(None)
    )

    # ---------------------------------------------------------
    # POLICY NUMBER
    # ---------------------------------------------------------

    .withColumn(
        "policy_number",
        when(
            upper(trim(col("policy_number"))).rlike(
                r"^POL_[0-9]{8}$"
            ),
            upper(trim(col("policy_number")))
        ).otherwise(None)
    )
)

display(
    silver_patients_df.select(
        "patient_id",
        "email",
        "phone_number",
        "policy_number"
    ).limit(50)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7362aaa-7fa6-4eba-9af6-bf75d1ae88f7)

In [22]:
# CELL 7 — Validate business date relationships


from pyspark.sql.functions import col, current_date, when

silver_patients_df = (
    silver_patients_df

    # Date of birth cannot be in the future
    .withColumn(
        "is_valid_date_of_birth",
        when(
            col("date_of_birth").isNull(),
            False
        ).when(
            col("date_of_birth") > current_date(),
            False
        ).otherwise(True)
    )

    # End date cannot be before start date
    .withColumn(
        "is_valid_date_range",
        when(
            col("start_date").isNotNull()
            & col("end_date").isNotNull()
            & (col("end_date") < col("start_date")),
            False
        ).otherwise(True)
    )
)

display(
    silver_patients_df.select(
        "patient_id",
        "date_of_birth",
        "start_date",
        "end_date",
        "is_valid_date_of_birth",
        "is_valid_date_range"
    ).limit(50)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7344b5e2-28d0-4b1b-89c2-98d2bc3e38c4)

In [23]:
# CELL 8 — CREATE REJECTION REASONS AND SPLIT VALID/QUARANTINE RECORDS
# Fully rerunnable: uses the cleaned and standardized values from Cells 1–7
# No tables are written in this cell
# Outputs:
#   valid_patients_df
#   quarantine_patients_df

from pyspark.sql.functions import (
    col,
    concat_ws,
    lit,
    when,
    current_timestamp,
    current_date,
    upper,
    trim
)

# ---------------------------------------------------------
# BUSINESS DATE LIMITS
# ---------------------------------------------------------

minimum_date_of_birth = lit("1901-01-01").cast("date")
minimum_operational_date = lit("2020-01-01").cast("date")

# Allows letters, accented letters, spaces, apostrophes and hyphens
name_pattern = r"^[A-Za-zÀ-ÖØ-öø-ÿ' -]{2,50}$"

# Placeholder values that must not enter Silver
invalid_text_values = [
    "UNKNOWN",
    "NO VALUE",
    "UNDEFINED",
    "N/A",
    "NA",
    "NONE",
    "NULL",
    "???",
    "??",
    "#"
]

# Approved insurance providers
# Comparison is case-insensitive
approved_insurance_providers = [
    "SUN LIFE",
    "MANULIFE",
    "CANADA LIFE",
    "BLUE CROSS",
    "DESJARDINS INSURANCE",
    "OHIP (ONTARIO)",
    "AHCIP (ALBERTA)",
    "MSP (BRITISH COLUMBIA)",
    "RAMQ (QUEBEC)",
    "MSI (NOVA SCOTIA)"
]

# ---------------------------------------------------------
# CREATE REJECTION REASONS
# ---------------------------------------------------------

silver_patients_df = (
    silver_patients_df
    .withColumn(
        "rejection_reason",
        concat_ws(
            "; ",

            # =================================================
            # PATIENT IDENTIFIER
            # =================================================

            when(
                col("patient_id").isNull(),
                lit("Invalid or missing patient_id")
            ),

            # =================================================
            # PATIENT NAME
            # =================================================

            when(
                col("first_name").isNull(),
                lit("Missing first_name")
            ),

            when(
                col("first_name").isNotNull()
                & upper(trim(col("first_name"))).isin(
                    invalid_text_values
                ),
                lit("Invalid first_name placeholder")
            ),

            when(
                col("first_name").isNotNull()
                & ~col("first_name").rlike(name_pattern),
                lit("Invalid first_name format")
            ),

            when(
                col("last_name").isNull(),
                lit("Missing last_name")
            ),

            when(
                col("last_name").isNotNull()
                & upper(trim(col("last_name"))).isin(
                    invalid_text_values
                ),
                lit("Invalid last_name placeholder")
            ),

            when(
                col("last_name").isNotNull()
                & ~col("last_name").rlike(name_pattern),
                lit("Invalid last_name format")
            ),

            # =================================================
            # DATE OF BIRTH
            # =================================================

            when(
                col("date_of_birth").isNull(),
                lit("Invalid or missing date_of_birth")
            ),

            when(
                col("date_of_birth").isNotNull()
                & (
                    col("date_of_birth")
                    < minimum_date_of_birth
                ),
                lit("Date of birth is unrealistically old")
            ),

            when(
                col("date_of_birth").isNotNull()
                & (
                    col("date_of_birth")
                    > current_date()
                ),
                lit("Date of birth is in the future")
            ),

            when(
                col("is_valid_date_of_birth") == False,
                lit("Invalid date of birth")
            ),

            # =================================================
            # GENDER AND ACTIVE STATUS
            # =================================================

            when(
                col("gender").isNull(),
                lit("Invalid or missing gender")
            ),

            when(
                col("is_active").isNull(),
                lit("Invalid or missing is_active")
            ),

            # =================================================
            # CONTACT INFORMATION
            # =================================================

            when(
                col("email").isNull(),
                lit("Invalid or missing email")
            ),

            when(
                col("phone_number").isNull(),
                lit("Invalid or missing phone_number")
            ),

            when(
                col("home_address").isNull()
                | upper(trim(col("home_address"))).isin(
                    invalid_text_values
                ),
                lit("Invalid or missing home_address")
            ),

            # =================================================
            # INSURANCE INFORMATION
            # =================================================

            when(
                col("insurance_provider").isNull(),
                lit("Missing insurance_provider")
            ),

            when(
                col("insurance_provider").isNotNull()
                & ~upper(
                    trim(col("insurance_provider"))
                ).isin(approved_insurance_providers),
                lit("Invalid insurance_provider")
            ),

            when(
                col("policy_number").isNull(),
                lit("Invalid or missing policy_number")
            ),

            # =================================================
            # COVERAGE DATES
            # =================================================

            when(
                col("start_date").isNull(),
                lit("Invalid or missing start_date")
            ),

            when(
                col("start_date").isNotNull()
                & (
                    col("start_date")
                    < minimum_operational_date
                ),
                lit("start_date is unrealistically old")
            ),

            when(
                col("start_date").isNotNull()
                & (
                    col("start_date")
                    > current_date()
                ),
                lit("start_date is in the future")
            ),

            when(
                col("end_date").isNotNull()
                & (
                    col("end_date")
                    < minimum_operational_date
                ),
                lit("end_date is unrealistically old")
            ),

            when(
                col("end_date").isNotNull()
                & (
                    col("end_date")
                    > current_date()
                ),
                lit("end_date is in the future")
            ),

            when(
                col("is_valid_date_range") == False,
                lit("End date is before start date")
            ),

            when(
                (col("is_active") == True)
                & col("end_date").isNotNull(),
                lit("Active patient contains an end_date")
            ),

            when(
                (col("is_active") == False)
                & col("end_date").isNull(),
                lit("Inactive patient is missing end_date")
            ),

            # =================================================
            # LAST UPDATED TIMESTAMP
            # =================================================

            when(
                col("last_updated_timestamp").isNull(),
                lit(
                    "Invalid or missing "
                    "last_updated_timestamp"
                )
            ),

            when(
                col("last_updated_timestamp").isNotNull()
                & (
                    col("last_updated_timestamp").cast("date")
                    < minimum_operational_date
                ),
                lit(
                    "last_updated_timestamp is "
                    "unrealistically old"
                )
            ),

            when(
                col("last_updated_timestamp").isNotNull()
                & (
                    col("last_updated_timestamp")
                    > current_timestamp()
                ),
                lit("last_updated_timestamp is in the future")
            ),

            when(
                col("last_updated_timestamp").isNotNull()
                & col("ingestion_timestamp").isNotNull()
                & (
                    col("last_updated_timestamp")
                    > col("ingestion_timestamp")
                ),
                lit(
                    "last_updated_timestamp is after "
                    "ingestion_timestamp"
                )
            ),

            when(
                col("last_updated_timestamp").isNotNull()
                & col("start_date").isNotNull()
                & (
                    col("last_updated_timestamp").cast("date")
                    < col("start_date")
                ),
                lit(
                    "last_updated_timestamp is before "
                    "start_date"
                )
            ),

            # =================================================
            # UNEXPECTED SOURCE DATA
            # =================================================

            when(
                col("unexpected_source_field").isNotNull(),
                lit("Unexpected source field contains data")
            )
        )
    )
)

# ---------------------------------------------------------
# SPLIT VALID AND QUARANTINE RECORDS
# ---------------------------------------------------------

valid_patients_df = (
    silver_patients_df
    .filter(col("rejection_reason") == "")
    .drop(
        "rejection_reason",
        "is_valid_date_of_birth",
        "is_valid_date_range"
    )
)

quarantine_patients_df = (
    silver_patients_df
    .filter(col("rejection_reason") != "")
    .withColumn(
        "rejection_timestamp",
        current_timestamp()
    )
    .drop(
        "is_valid_date_of_birth",
        "is_valid_date_range"
    )
)

# ---------------------------------------------------------
# RECONCILIATION
# ---------------------------------------------------------

total_count = silver_patients_df.count()
valid_count = valid_patients_df.count()
quarantine_count = quarantine_patients_df.count()

reconciled_count = valid_count + quarantine_count

print("Total rows:", total_count)
print("Valid rows:", valid_count)
print("Quarantine rows:", quarantine_count)
print("Reconciled rows:", reconciled_count)

if total_count != reconciled_count:
    raise ValueError(
        f"Validation reconciliation failed. "
        f"Total={total_count}, "
        f"Valid+Quarantine={reconciled_count}"
    )

if valid_count == 0:
    raise ValueError(
        "Validation produced zero valid patient rows. "
        "Review the validation rules before continuing."
    )

print(
    "SUCCESS: All patient rows were classified "
    "as valid or quarantined."
)

# ---------------------------------------------------------
# DISPLAY REJECTED SAMPLE
# ---------------------------------------------------------

display(
    quarantine_patients_df.select(
        "patient_id",
        "first_name",
        "last_name",
        "date_of_birth",
        "gender",
        "email",
        "phone_number",
        "home_address",
        "insurance_provider",
        "policy_number",
        "is_active",
        "start_date",
        "end_date",
        "last_updated_timestamp",
        "unexpected_source_field",
        "rejection_reason"
    ).limit(50)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 25, Finished, Available, Finished, False)

Total rows: 540743
Valid rows: 143
Quarantine rows: 540600
Reconciled rows: 540743
SUCCESS: All patient rows were classified as valid or quarantined.


SynapseWidget(Synapse.DataFrame, 19906d4e-be20-4d2e-bd92-68f7e678f217)

In [24]:
# CELL 9 — Deduplicate valid patient records
# Keep the latest record and send duplicates to quarantine

from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number,
    col,
    lit,
    current_timestamp
)

patient_window = (
    Window
    .partitionBy("patient_id")
    .orderBy(
        col("last_updated_timestamp").desc_nulls_last(),
        col("ingestion_timestamp").desc_nulls_last()
    )
)

ranked_patients_df = (
    valid_patients_df
    .withColumn(
        "duplicate_rank",
        row_number().over(patient_window)
    )
)

# Records eligible for Silver
deduplicated_patients_df = (
    ranked_patients_df
    .filter(col("duplicate_rank") == 1)
    .drop("duplicate_rank")
)

# Duplicate records sent to quarantine
duplicate_patients_df = (
    ranked_patients_df
    .filter(col("duplicate_rank") > 1)
    .drop("duplicate_rank")
    .withColumn(
        "rejection_reason",
        lit("Duplicate patient_id")
    )
    .withColumn(
        "rejection_timestamp",
        current_timestamp()
    )
)

# Combine validation failures and duplicate failures
final_quarantine_patients_df = (
    quarantine_patients_df
    .unionByName(
        duplicate_patients_df,
        allowMissingColumns=True
    )
)

valid_before_dedup_count = valid_patients_df.count()
silver_candidate_count = deduplicated_patients_df.count()
duplicate_count = duplicate_patients_df.count()

print("Valid rows before deduplication:", valid_before_dedup_count)
print("Silver candidate rows:", silver_candidate_count)
print("Duplicate rows:", duplicate_count)
print(
    "Final quarantine rows:",
    final_quarantine_patients_df.count()
)

if valid_before_dedup_count != silver_candidate_count + duplicate_count:
    raise ValueError(
        f"Deduplication reconciliation failed. "
        f"Valid={valid_before_dedup_count}, "
        f"Silver candidates + duplicates="
        f"{silver_candidate_count + duplicate_count}"
    )

print("SUCCESS: Patient records were deduplicated correctly.")

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 26, Finished, Available, Finished, False)

Valid rows before deduplication: 143
Silver candidate rows: 135
Duplicate rows: 8
Final quarantine rows: 540608
SUCCESS: Patient records were deduplicated correctly.


In [25]:
# CELL 10 — OVERWRITE CLEAN SILVER AND QUARANTINE PATIENT TABLES
# Fully rerunnable: rebuilds both tables from current notebook results

from pyspark.sql.functions import (
    col,
    sha2,
    concat_ws,
    coalesce,
    lit,
    current_timestamp,
    row_number
)
from pyspark.sql.window import Window


silver_table_name = "silver_patients"
quarantine_table_name = "quarantine_patients"


# PART 1 — PREPARE CLEAN SILVER SOURCE

patient_hash_columns = [
    "patient_id",
    "first_name",
    "last_name",
    "date_of_birth",
    "gender",
    "email",
    "phone_number",
    "home_address",
    "insurance_provider",
    "policy_number",
    "is_active",
    "start_date",
    "end_date",
    "last_updated_timestamp",
    "unexpected_source_field"
]

silver_patients_source_df = (
    deduplicated_patients_df

    .withColumn(
        "row_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in patient_hash_columns
                ]
            ),
            256
        )
    )

    .withColumn(
        "silver_processing_timestamp",
        current_timestamp()
    )

    # Final safety check:
    # one patient_id must produce only one Silver row
    .dropDuplicates(["patient_id"])
)


silver_candidate_count = (
    deduplicated_patients_df.count()
)

silver_source_count = (
    silver_patients_source_df.count()
)

print("Silver candidate rows:", silver_candidate_count)
print("Silver source rows:", silver_source_count)

if silver_candidate_count != silver_source_count:
    raise ValueError(
        f"Silver source reconciliation failed. "
        f"Candidates={silver_candidate_count}, "
        f"Source={silver_source_count}"
    )


# PART 2 — OVERWRITE SILVER PATIENTS

(
    silver_patients_source_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table_name)
)

final_silver_count = (
    spark.table(silver_table_name).count()
)

print("Final Silver rows:", final_silver_count)

if final_silver_count != silver_source_count:
    raise ValueError(
        f"Silver write validation failed. "
        f"Source={silver_source_count}, "
        f"Table={final_silver_count}"
    )

print(
    "SUCCESS: Silver patients table was rebuilt "
    "using only valid and deduplicated records."
)


# PART 3 — PREPARE QUARANTINE SOURCE
# Preserve every rejected Bronze row, including identical rows

quarantine_hash_columns = [
    "batch_id",
    "source_file_name",
    "patient_id",
    "first_name",
    "last_name",
    "date_of_birth",
    "gender",
    "email",
    "phone_number",
    "home_address",
    "insurance_provider",
    "policy_number",
    "is_active",
    "start_date",
    "end_date",
    "last_updated_timestamp",
    "unexpected_source_field",
    "rejection_reason"
]

quarantine_with_base_hash_df = (
    final_quarantine_patients_df

    .withColumn(
        "quarantine_base_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in quarantine_hash_columns
                ]
            ),
            256
        )
    )
)


# Assign a stable occurrence number to identical rejected rows
quarantine_occurrence_window = (
    Window
    .partitionBy("quarantine_base_hash")
    .orderBy(
        col("ingestion_timestamp").asc_nulls_last(),
        col("patient_id").asc_nulls_last(),
        col("first_name").asc_nulls_last(),
        col("last_name").asc_nulls_last()
    )
)


quarantine_patients_source_df = (
    quarantine_with_base_hash_df

    .withColumn(
        "quarantine_occurrence_number",
        row_number().over(
            quarantine_occurrence_window
        )
    )

    .withColumn(
        "quarantine_record_id",
        sha2(
            concat_ws(
                "||",
                col("quarantine_base_hash"),
                col(
                    "quarantine_occurrence_number"
                ).cast("string")
            ),
            256
        )
    )

    .withColumn(
        "quarantine_processing_timestamp",
        current_timestamp()
    )

    .drop("quarantine_base_hash")
)


quarantine_candidate_count = (
    final_quarantine_patients_df.count()
)

quarantine_source_count = (
    quarantine_patients_source_df.count()
)

print(
    "Quarantine candidate rows:",
    quarantine_candidate_count
)

print(
    "Quarantine source rows:",
    quarantine_source_count
)

if quarantine_candidate_count != quarantine_source_count:
    raise ValueError(
        f"Quarantine source reconciliation failed. "
        f"Candidates={quarantine_candidate_count}, "
        f"Source={quarantine_source_count}"
    )


# PART 4 — OVERWRITE QUARANTINE PATIENTS

(
    quarantine_patients_source_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quarantine_table_name)
)

final_quarantine_count = (
    spark.table(quarantine_table_name).count()
)

print(
    "Final Quarantine rows:",
    final_quarantine_count
)

if final_quarantine_count != quarantine_source_count:
    raise ValueError(
        f"Quarantine write validation failed. "
        f"Source={quarantine_source_count}, "
        f"Table={final_quarantine_count}"
    )

print(
    "SUCCESS: Quarantine patients table was rebuilt "
    "without removing rejected source rows."
)


# PART 5 — FINAL WRITE RECONCILIATION

bronze_count = bronze_patients_df.count()

physical_reconciled_count = (
    final_silver_count
    + final_quarantine_count
)

print("Bronze rows:", bronze_count)
print("Final Silver rows:", final_silver_count)
print("Final Quarantine rows:", final_quarantine_count)
print("Reconciled rows:", physical_reconciled_count)

if bronze_count != physical_reconciled_count:
    raise ValueError(
        f"Physical table reconciliation failed. "
        f"Bronze={bronze_count}, "
        f"Silver+Quarantine={physical_reconciled_count}"
    )

print(
    "SUCCESS: Every Bronze patient row is accounted "
    "for in Silver or Quarantine."
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 27, Finished, Available, Finished, False)

Silver candidate rows: 135
Silver source rows: 135
Final Silver rows: 135
SUCCESS: Silver patients table was rebuilt using only valid and deduplicated records.
Quarantine candidate rows: 540608
Quarantine source rows: 540608
Final Quarantine rows: 540608
SUCCESS: Quarantine patients table was rebuilt without removing rejected source rows.
Bronze rows: 540743
Final Silver rows: 135
Final Quarantine rows: 540608
Reconciled rows: 540743
SUCCESS: Every Bronze patient row is accounted for in Silver or Quarantine.


In [26]:
# CELL 11 — VERIFY CURRENT PATIENT BATCH RECONCILIATION
# Fully rerunnable: validates both dataframe and physical table results

from pyspark.sql.functions import col

# Get exactly one current batch_id
batch_ids = [
    row["batch_id"]
    for row in (
        bronze_patients_df
        .select("batch_id")
        .where(col("batch_id").isNotNull())
        .distinct()
        .collect()
    )
]

if len(batch_ids) != 1:
    raise ValueError(
        f"Expected exactly one batch_id, but found: {batch_ids}"
    )

current_batch_id = batch_ids[0]

# Current Bronze batch rows
bronze_batch_count = (
    bronze_patients_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

# Expected dataframe results
silver_candidate_count = (
    deduplicated_patients_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

quarantine_candidate_count = (
    final_quarantine_patients_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

dataframe_reconciled_count = (
    silver_candidate_count
    + quarantine_candidate_count
)

print("Batch ID:", current_batch_id)
print("Bronze batch rows:", bronze_batch_count)
print("Silver candidate rows:", silver_candidate_count)
print(
    "Quarantine candidate rows:",
    quarantine_candidate_count
)
print(
    "Dataframe reconciled rows:",
    dataframe_reconciled_count
)

if bronze_batch_count != dataframe_reconciled_count:
    raise ValueError(
        f"Dataframe reconciliation failed. "
        f"Bronze={bronze_batch_count}, "
        f"Silver candidates + Quarantine candidates="
        f"{dataframe_reconciled_count}"
    )

print("SUCCESS: Dataframe reconciliation passed.")

# Read persisted tables
silver_patients_check_df = spark.table(
    "silver_patients"
)

quarantine_patients_check_df = spark.table(
    "quarantine_patients"
)

# Actual physical rows for current batch
final_silver_batch_count = (
    silver_patients_check_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

final_quarantine_batch_count = (
    quarantine_patients_check_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

physical_reconciled_count = (
    final_silver_batch_count
    + final_quarantine_batch_count
)

print(
    "Final Silver table rows for batch:",
    final_silver_batch_count
)

print(
    "Final Quarantine table rows for batch:",
    final_quarantine_batch_count
)

print(
    "Physical reconciled rows:",
    physical_reconciled_count
)

if final_silver_batch_count != silver_candidate_count:
    raise ValueError(
        f"Silver table validation failed. "
        f"Expected={silver_candidate_count}, "
        f"Actual={final_silver_batch_count}"
    )

if (
    final_quarantine_batch_count
    != quarantine_candidate_count
):
    raise ValueError(
        f"Quarantine table validation failed. "
        f"Expected={quarantine_candidate_count}, "
        f"Actual={final_quarantine_batch_count}"
    )

if bronze_batch_count != physical_reconciled_count:
    raise ValueError(
        f"Physical reconciliation failed. "
        f"Bronze={bronze_batch_count}, "
        f"Silver + Quarantine="
        f"{physical_reconciled_count}"
    )

print(
    "SUCCESS: Every Bronze patient row is accounted "
    "for in the persisted Silver or Quarantine table."
)

display(
    silver_patients_check_df
    .filter(col("batch_id") == current_batch_id)
    .limit(20)
)

display(
    quarantine_patients_check_df
    .filter(col("batch_id") == current_batch_id)
    .select(
        "quarantine_record_id",
        "quarantine_occurrence_number",
        "patient_id",
        "date_of_birth",
        "start_date",
        "end_date",
        "rejection_reason",
        "rejection_timestamp"
    )
    .limit(20)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 28, Finished, Available, Finished, False)

Batch ID: patients_batch_001
Bronze batch rows: 540743
Silver candidate rows: 135
Quarantine candidate rows: 540608
Dataframe reconciled rows: 540743
SUCCESS: Dataframe reconciliation passed.
Final Silver table rows for batch: 135
Final Quarantine table rows for batch: 540608
Physical reconciled rows: 540743
SUCCESS: Every Bronze patient row is accounted for in the persisted Silver or Quarantine table.


SynapseWidget(Synapse.DataFrame, a07150f4-3fb2-4f2d-9850-42dd2bc36f31)

SynapseWidget(Synapse.DataFrame, ce2983fb-3ffa-4b6e-b395-804a6984b67b)

In [27]:
# CELL 12 — UPSERT PATIENTS AUDIT RECORD
# Rerunnable: one audit row per batch_id + dataset_name

from pyspark.sql.functions import col

dataset_name = "patients"

# Get the current batch_id from Bronze
batch_ids = [
    row["batch_id"]
    for row in (
        bronze_patients_df
        .select("batch_id")
        .where(col("batch_id").isNotNull())
        .distinct()
        .collect()
    )
]

if len(batch_ids) != 1:
    raise ValueError(
        f"Expected exactly one batch_id, but found: {batch_ids}"
    )

current_batch_id = batch_ids[0]

# Count only the current processing batch
rows_received = (
    bronze_patients_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

rows_written = (
    deduplicated_patients_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

rejected_rows = (
    final_quarantine_patients_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

if rows_received != rows_written + rejected_rows:
    raise ValueError(
        f"Audit reconciliation failed. "
        f"Received={rows_received}, "
        f"Written+Rejected={rows_written + rejected_rows}"
    )

print("Batch ID:", current_batch_id)
print("Rows received:", rows_received)
print("Rows written:", rows_written)
print("Rejected rows:", rejected_rows)

audit_source_df = spark.createDataFrame(
    [
        (
            current_batch_id,
            dataset_name,
            rows_received,
            rows_written,
            rejected_rows,
            "COMPLETED"
        )
    ],
    """
    batch_id string,
    dataset_name string,
    rows_received long,
    rows_written long,
    rejected_rows long,
    status string
    """
)

audit_source_df.createOrReplaceTempView(
    "patients_audit_source"
)

spark.sql("""
MERGE INTO healthcare_control_lakehouse.dbo.audit_table AS target

USING (
    SELECT
        batch_id,
        dataset_name,
        current_timestamp() AS start_timestamp,
        current_timestamp() AS end_timestamp,
        rows_received,
        rows_written,
        rejected_rows,
        status
    FROM patients_audit_source
) AS source

ON target.batch_id = source.batch_id
AND target.dataset_name = source.dataset_name

WHEN MATCHED THEN UPDATE SET
    target.end_timestamp = source.end_timestamp,
    target.rows_received = source.rows_received,
    target.rows_written = source.rows_written,
    target.rejected_rows = source.rejected_rows,
    target.status = source.status

WHEN NOT MATCHED THEN INSERT (
    batch_id,
    dataset_name,
    start_timestamp,
    end_timestamp,
    rows_received,
    rows_written,
    rejected_rows,
    status
)
VALUES (
    source.batch_id,
    source.dataset_name,
    source.start_timestamp,
    source.end_timestamp,
    source.rows_received,
    source.rows_written,
    source.rejected_rows,
    source.status
)
""")

print("Patients audit record inserted or updated successfully.")

display(
    spark.sql(f"""
        SELECT *
        FROM healthcare_control_lakehouse.dbo.audit_table
        WHERE batch_id = '{current_batch_id}'
          AND dataset_name = '{dataset_name}'
    """)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 29, Finished, Available, Finished, False)

Batch ID: patients_batch_001
Rows received: 540743
Rows written: 135
Rejected rows: 540608
Patients audit record inserted or updated successfully.


SynapseWidget(Synapse.DataFrame, e4858f23-21cd-42df-be0a-b46e1c63efc9)

In [28]:
# CEll 13 CONTROL TABLE — Insert or update patients configuration

spark.sql("""
MERGE INTO healthcare_control_lakehouse.dbo.control_table AS target

USING (
    SELECT
        1 AS control_id,
        'patients' AS dataset_name,
        'Files/raw/patients.csv' AS source_path,
        'bronze_patients' AS bronze_table_name,
        'silver_patients' AS silver_table_name,
        true AS is_active
) AS source

ON target.dataset_name = source.dataset_name

WHEN MATCHED THEN UPDATE SET
    target.source_path = source.source_path,
    target.bronze_table_name = source.bronze_table_name,
    target.silver_table_name = source.silver_table_name,
    target.is_active = source.is_active

WHEN NOT MATCHED THEN INSERT (
    control_id,
    dataset_name,
    source_path,
    bronze_table_name,
    silver_table_name,
    is_active
)
VALUES (
    source.control_id,
    source.dataset_name,
    source.source_path,
    source.bronze_table_name,
    source.silver_table_name,
    source.is_active
)
""")

print("Patients control record inserted or updated.")

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 30, Finished, Available, Finished, False)

Patients control record inserted or updated.


In [29]:
# CELL 14 — UPSERT PATIENTS FILE-TRACKING RECORD
# Rerunnable and safe for parallel notebook execution

from pyspark.sql.functions import (
    col,
    concat_ws,
    coalesce,
    lit,
    xxhash64
)

file_tracking_table = (
    "healthcare_control_lakehouse.dbo.file_tracking_table"
)

dataset_name = "patients"

# Get exactly one batch_id from the current Bronze data
batch_ids = [
    row["batch_id"]
    for row in (
        bronze_patients_df
        .select("batch_id")
        .where(col("batch_id").isNotNull())
        .distinct()
        .collect()
    )
]

if len(batch_ids) != 1:
    raise ValueError(
        f"Expected exactly one batch_id, but found: {batch_ids}"
    )

current_batch_id = batch_ids[0]

# Get exactly one source file for the current batch
source_files = [
    row["source_file_name"]
    for row in (
        bronze_patients_df
        .filter(col("batch_id") == current_batch_id)
        .select("source_file_name")
        .where(col("source_file_name").isNotNull())
        .distinct()
        .collect()
    )
]

if len(source_files) != 1:
    raise ValueError(
        f"Expected exactly one source file for "
        f"batch_id={current_batch_id}, but found: {source_files}"
    )

current_source_file = source_files[0]

# Create the file-tracking source record
file_tracking_source_df = spark.createDataFrame(
    [
        (
            current_source_file,
            dataset_name,
            current_batch_id,
            "COMPLETED"
        )
    ],
    """
    source_file_name string,
    dataset_name string,
    batch_id string,
    processing_status string
    """
)

# Generate a deterministic LONG ID
# The same file + dataset + batch always generates the same ID
file_tracking_source_df = (
    file_tracking_source_df
    .withColumn(
        "file_tracking_id",
        xxhash64(
            concat_ws(
                "||",
                coalesce(col("source_file_name"), lit("")),
                coalesce(col("dataset_name"), lit("")),
                coalesce(col("batch_id"), lit(""))
            )
        )
    )
    .select(
        "file_tracking_id",
        "source_file_name",
        "dataset_name",
        "batch_id",
        "processing_status"
    )
    .dropDuplicates(
        [
            "source_file_name",
            "dataset_name",
            "batch_id"
        ]
    )
)

file_tracking_source_df.createOrReplaceTempView(
    "patients_file_tracking_source"
)

spark.sql(f"""
MERGE INTO {file_tracking_table} AS target

USING (
    SELECT
        file_tracking_id,
        source_file_name,
        dataset_name,
        batch_id,
        current_timestamp() AS processed_timestamp,
        processing_status
    FROM patients_file_tracking_source
) AS source

ON target.source_file_name = source.source_file_name
AND target.dataset_name = source.dataset_name
AND target.batch_id = source.batch_id

WHEN MATCHED THEN UPDATE SET
    target.processed_timestamp = source.processed_timestamp,
    target.processing_status = source.processing_status

WHEN NOT MATCHED THEN INSERT (
    file_tracking_id,
    source_file_name,
    dataset_name,
    batch_id,
    processed_timestamp,
    processing_status
)
VALUES (
    source.file_tracking_id,
    source.source_file_name,
    source.dataset_name,
    source.batch_id,
    source.processed_timestamp,
    source.processing_status
)
""")

print("Patients file-tracking record inserted or updated successfully.")
print("Batch ID:", current_batch_id)
print("Dataset:", dataset_name)
print("Source file:", current_source_file)

display(
    spark.sql(f"""
        SELECT *
        FROM {file_tracking_table}
        WHERE source_file_name = '{current_source_file}'
          AND dataset_name = '{dataset_name}'
          AND batch_id = '{current_batch_id}'
    """)
)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 31, Finished, Available, Finished, False)

Patients file-tracking record inserted or updated successfully.
Batch ID: patients_batch_001
Dataset: patients
Source file: patients.csv


SynapseWidget(Synapse.DataFrame, c3fa7eb2-7994-488f-ba43-ba24bd6b71e2)

In [30]:
silver_df = (
    spark.read
    .format("delta")
    .load(
        "abfss://healthcare_streaming_platform@onelake.dfs.fabric.microsoft.com/"
        "healthcare_streaming_lakehouse.Lakehouse/Tables/dbo/silver_patients"
    )
)

silver_df.show(100, truncate=False)

StatementMeta(, 0ae2b733-1dbc-4c35-ad3a-469b7b1e0e4f, 32, Finished, Available, Finished, False)

+----------+----------+----------+-------------+------+----------------------------+----------------+---------------------------------------------------------------+----------------------+-------------+---------+----------+----------+--------------------------+-----------------------+------------------+----------------+--------------------------+----------------------------------------------------------------+---------------------------+
|patient_id|first_name|last_name |date_of_birth|gender|email                       |phone_number    |home_address                                                   |insurance_provider    |policy_number|is_active|start_date|end_date  |last_updated_timestamp    |unexpected_source_field|batch_id          |source_file_name|ingestion_timestamp       |row_hash                                                        |silver_processing_timestamp|
+----------+----------+----------+-------------+------+----------------------------+----------------+---------------